# Part 15 — Adaptive RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-15-adaptive-rag/adaptive_rag.ipynb)

*A small complexity classifier routes each query to the pipeline it actually needs — and closes the Frontier Track.*

📖 Read the essay: https://www.mefby.com/essays/adaptive-rag

🐍 Companion script: `adaptive_rag.py`

## What you'll build

For nine parts we built pipelines that run the **same** path every time: retrieve once, stuff the chunks into a prompt, generate. Part 10 gave that pipeline a brain (control flow). This part gives it a **conductor**.

A greeting needs no retrieval. A single fact needs one lookup. A comparison needs several. A fixed pipeline either **over-serves** the easy queries (paying for retrieval they don't need) or **under-serves** the hard ones (one lookup can't answer a comparison). **Adaptive RAG** reads each query first and routes it to one of three pipelines we already know how to build:

- **none** → answer directly, no retrieval (greeting / the model already knows);
- **single** → one retrieve → generate (the **Part 6** pipeline);
- **multi** → decompose → retrieve per part → synthesize (the **Part 10** shape).

Concretely you'll build, cell by cell:

- the support knowledge base carried over from Parts 6–12 (refunds, the **E-4042** error code, shipping, warranty);
- `classify_complexity(query)` — a tiny, fully legible rule/keyword classifier returning `none` / `single` / `multi`;
- an **embedder** with a transparent offline fallback (the Part 2 pattern) and a one-store **retriever** (Parts 4 + 6);
- the three route handlers and the `route()` **conductor** that dispatches among them;
- a demo that routes six queries (two per class) and prints the chosen path and answer for each.

This is route-by-**complexity**. It is distinct from **Part 8** (which *transforms* the query to retrieve better) and from **Part 10**'s source routing (which routes by *which* knowledge source, not by how hard the question is).

> **Runs with no network and no API key.** `numpy` is the only required dependency; every code cell executes offline against the transparent deterministic stand-ins. If `sentence-transformers` is installed and a model is cached, the real embedder runs locally (still no network) and only the cosine scores shift — the route labels and the none/single/multi tally are identical either way. The real classifier / embedder / LLM paths are shown as the headline (the code you'd actually ship), each behind a `try/except` that prints a clear label when it falls back.

## Setup

Two imports do the work: `re` for the classifier's keyword rules and tokenizing, and `numpy` for the vector math. `os` lets us check for an API key without ever requiring one. Nothing here touches the network.

In [ ]:
import os
import re

import numpy as np

print("numpy", np.__version__, "— ready (no network, no API key required)")

## Step 0 — The support knowledge base

We reuse the same store-policy corpus that ran through Parts 6–12, so this part feels continuous: refunds, the **E-4042** error code, shipping, and warranty. The docs are short, so each doc is its own chunk (Part 5). The router will reach into this index only for the `single` and `multi` routes — never for `none`.

In [ ]:
KNOWLEDGE_BASE = [
    "Refunds are accepted within 30 days of purchase, provided the item is unused and in its original packaging.",
    "To start a return, email support@example.com with your order number. Refunds are processed within five business days of us receiving the item.",
    "Error E-4042 means the payment was declined by the bank; ask the customer to retry with a different card or contact their bank.",
    "Standard shipping takes 3 to 5 business days. Express shipping arrives the next business day.",
    "All electronics include a one-year limited warranty covering manufacturing defects.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} support chunks.")
for i, doc in enumerate(KNOWLEDGE_BASE):
    print(f"  [{i}] {doc[:60]}...")

## Step 1 — The complexity classifier: the heart of Adaptive RAG

This is the one component Adaptive RAG adds. It reads **one** query and returns the route it deserves, by inspecting cheap, transparent signals:

- very short, a greeting, or chit-chat → **none** (no retrieval);
- a comparison, conjunction, or multi-question → **multi** (decompose);
- everything else → **single** (one lookup).

A production system swaps this for a small **trained** classifier (we show that path next). The rules here are deliberately legible so you can see exactly *why* a query routes the way it does — the whole teaching point of the part. Note what signals complexity: a comparative word (`compare`, `versus`, `difference between`), an explicit conjunction-of-tasks (`and then`, `both`, `each of`), or more than one question mark.

In [ ]:
def classify_complexity(query: str) -> str:
    """Route a query by complexity: 'none' (no retrieval needed),
    'single' (one retrieve->generate), or 'multi' (decompose, multi-step).
    Deterministic rule/keyword classifier; a real system would use a small
    trained classifier, shown here as the fallback path."""
    q = query.lower().strip()
    if re.search(r"\b(hi|hello|thanks|who are you)\b", q) or len(q.split()) <= 2:
        return "none"
    multi_signals = ("compare", "versus", " vs ", "difference between",
                     " and then", "across", "each of", "both", "trade-off")
    if any(s in q for s in multi_signals) or q.count("?") > 1:
        return "multi"
    return "single"


# See the three bands fire on one query each:
for q in ["hi there",
          "What is our refund window?",
          "Compare the refund window and the warranty period."]:
    print(f"  {classify_complexity(q):<7} <- {q}")

### The real classifier path (headline; never run offline)

In production the router is usually a small **trained** text classifier (a fine-tuned encoder or a tiny LLM) returning one of the three route labels. Here is the shape, loaded behind a `try/except` so this notebook still runs with no model and no network: if the load fails, we keep the deterministic rules above and print a clear banner. **The real path is the headline; the rules keep the cell executable.**

In [ ]:
def load_real_classifier():
    """Try to load a small trained complexity classifier; None if unavailable."""
    try:
        # Intended path: a tiny text-classification model returning one of the
        # three route labels. Names/APIs move fast; check current docs.
        from transformers import pipeline  # noqa: F401

        clf = pipeline("text-classification", model="adaptive-rag-router")

        def classify(query: str) -> str:
            label = clf(query)[0]["label"].lower()
            return label if label in {"none", "single", "multi"} else classify_complexity(query)

        return classify
    except Exception as exc:  # not installed / no weights / offline
        print(f"[classify] trained classifier unavailable ({type(exc).__name__}); "
              "using deterministic rule/keyword classifier (offline default)")
        return None


_real_classify = load_real_classifier()
classify = _real_classify if _real_classify is not None else classify_complexity
if _real_classify is not None:
    print("[classify] using trained complexity classifier")

## Step 2 — Embeddings, with a transparent offline fallback

The `single` and `multi` routes need real retrieval, so they need an embedder. The headline is `sentence-transformers`, exactly as in Part 6 — a real model that learned to place similar meanings near each other. We wrap the load in `try/except` so the notebook still runs with no model and no network; if it can't load, we drop to a deterministic hashing stand-in.

The stand-in mimics the **interface** (text → fixed-length dense unit vector) but not the meaning: a hash has no idea "refund window" and "refunds within 30 days" are cousins. To keep the demo honest offline we give it two crude hints — drop stop-words so *content* words dominate, and strip a trailing plural `s` so `refunds` and `refund` hash to the same bucket. A real model needs neither.

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+")

# Grammatical words the offline embedder ignores so a query's CONTENT words
# (refund, window, warranty) drive the cosine instead of "what / is / our".
_STOPWORDS = {
    "what", "is", "are", "the", "of", "a", "an", "for", "on", "in", "to", "how",
    "do", "does", "and", "my", "i", "there", "with", "your", "our", "between",
    "explain", "compare", "difference",
}


def _stem(tok):
    """Crudest possible stemmer: drop a trailing plural 's' so 'refunds' and
    'refund' hash to the same bucket. A real model needs no such hint."""
    return tok[:-1] if len(tok) > 3 and tok.endswith("s") else tok


def _tokens(text):
    """Lowercase word/number tokens -- the level the essay stays at."""
    return _TOKEN_RE.findall(text.lower())

### The offline stand-in: a deterministic hashing embedder

Hash each **content** token into a fixed-width vector and accumulate, then L2-normalize. Same text → same vector, always. Texts that share content words land closer (shared tokens push the same dimensions the same way). `normalize=True` means a later dot product reads straight off as cosine similarity (the Part 3 trick).

In [ ]:
class _HashingEmbedder:
    """Deterministic, model-free stand-in for a sentence embedder (Part 2)."""

    def __init__(self, dim=256):
        self.dim = dim

    def encode(self, texts, normalize_embeddings=True):
        vecs = np.zeros((len(texts), self.dim), dtype=np.float64)
        for r, text in enumerate(texts):
            toks = [_stem(t) for t in _tokens(text) if t not in _STOPWORDS]
            for tok in toks:
                # Stable hash -> bucket; sign spreads collisions. Deterministic
                # across runs (no dependence on PYTHONHASHSEED).
                h = 0
                for ch in tok:
                    h = (h * 131 + ord(ch)) & 0xFFFFFFFF
                vecs[r, h % self.dim] += 1.0 if (h >> 1) & 1 else -1.0
        if normalize_embeddings:
            norms = np.linalg.norm(vecs, axis=1, keepdims=True)
            norms[norms == 0] = 1.0
            vecs = vecs / norms
        return vecs


def load_real_embedder():
    """Try the real model first; fall back transparently if unavailable."""
    try:
        from sentence_transformers import SentenceTransformer  # REAL path
        model = SentenceTransformer("all-MiniLM-L6-v2")         # 384 dims
        print("[embed] using sentence-transformers (all-MiniLM-L6-v2)")
        return model, True
    except Exception as exc:  # not installed / no weights / offline
        print(f"[embed] sentence-transformers unavailable ({type(exc).__name__}); "
              "using deterministic hashing fallback")
        return _HashingEmbedder(), False


_embedder, _real_embed = load_real_embedder()


def embed(texts):
    return np.asarray(_embedder.encode(texts, normalize_embeddings=True))


# Quick sanity check: text in -> unit vectors out, in one shared space.
_v = embed(["refund window", "warranty period"])
print(f"embed() returns shape {_v.shape}; row norms ~= {np.round(np.linalg.norm(_v, axis=1), 3).tolist()}")

## Step 3 — A tiny vector store + retriever

A "vector store" is just the chunks kept side by side with a matrix of their vectors (Part 4). `retrieve()` embeds the query with the **same** model, scores every chunk by cosine similarity (a dot product, since the vectors are unit length), and keeps the top-k (Part 3 + Part 4). We default to `k=1` here so each route returns its single best chunk.

In [ ]:
class VectorStore:
    def __init__(self, corpus, embed):
        self.chunks = list(corpus)
        self.vectors = embed(self.chunks)         # (n_chunks, dim)
        self._embed = embed

    def retrieve(self, query, k=1):
        q = self._embed([query])[0]               # same model as the chunks
        scores = self.vectors @ q                 # cosine sim (unit vectors)
        top = np.argsort(-scores)[:k]             # indices of k highest scores
        return [(self.chunks[i], float(scores[i])) for i in top]


store = VectorStore(KNOWLEDGE_BASE, embed)

# Peek: a refund question should pull the refund chunk to the top.
for text, score in store.retrieve("What is our refund window?", k=1):
    print(f"  {score:+.3f}  {text[:60]}...")

## Step 4 — Generation: a grounded extractive stand-in

The intended path is one swappable hosted-LLM call (Part 6's `generate()`), kept behind the same grounding instruction so the model answers **only** from context. Offline we use an extractive stand-in that quotes the single best-retrieved chunk, so the answer is visibly tied to a source and the whole demo runs with no model.

In [ ]:
def generate(retrieved):
    """Grounded extractive generate(): quote the single best-retrieved chunk."""
    if not retrieved:
        return "I don't know based on the available sources."
    best_text, _score = retrieved[0]
    return f"Based on the retrieved policy: {best_text}"


print(generate(store.retrieve("How do I fix the E-4042 error?")))

## Step 5 — The three route handlers

Each route is a pipeline we already built. Before the conductor can dispatch, we define the pieces:

- the **none** route answers directly, with no retrieval and no embedder call;
- the **single** route is the Part 6 retrieve → generate pipeline (handled inline in `route`);
- the **multi** route needs to **decompose** the query into independently-retrievable sub-queries, on the same surface signals the classifier keyed on (comparisons, conjunctions, multiple questions), then retrieve per part and synthesize.

First the `none` route's canned replies and the decomposition helper.

In [ ]:
# Canned no-retrieval replies for the 'none' route. A real system would let the
# LLM answer directly here; offline we template so it stays deterministic.
_DIRECT_REPLIES = {
    "greeting": ("Hi! I'm the support assistant. Ask me about refunds, shipping, "
                 "the E-4042 error, or your warranty and I'll look it up."),
    "thanks": "You're welcome! Anything else about your order or our policies?",
    "identity": ("I'm the support assistant for this store. I answer from our "
                 "policy docs -- refunds, shipping, errors, and warranty."),
}


def _direct_answer(query: str) -> str:
    """The 'none' route: answer without touching the index."""
    q = query.lower()
    if "thank" in q:
        return _DIRECT_REPLIES["thanks"]
    if "who are you" in q or "what are you" in q:
        return _DIRECT_REPLIES["identity"]
    return _DIRECT_REPLIES["greeting"]

In [ ]:
# Split a 'multi' query into sub-queries on the same surface signals the
# classifier keyed on: comparisons, conjunctions, and multiple questions.
_DECOMPOSE_RE = re.compile(
    r"\s+and then\s+|\s+versus\s+|\s+vs\.?\s+|\s+difference between\s+"
    r"|,?\s+and\s+|\s*\?\s*",
    flags=re.IGNORECASE,
)
# Lead-in phrases to strip from each sub-query so retrieval sees the content.
_LEADINS = ("compare the", "compare", "explain the", "explain",
            "the difference between", "difference between", "what is")


def decompose(query: str) -> list:
    """Break a complex query into smaller, independently-retrievable parts."""
    raw = [p.strip(" .") for p in _DECOMPOSE_RE.split(query) if p.strip(" .")]
    subs = []
    for part in raw:
        cleaned = part
        low = cleaned.lower()
        for lead in _LEADINS:
            if low.startswith(lead):
                cleaned = cleaned[len(lead):].strip()
                break
        # Drop fragments that are nothing but connective/stop words (e.g. a
        # leftover "the difference") -- they have no content to retrieve on.
        if cleaned and any(t not in _STOPWORDS for t in _tokens(cleaned)):
            subs.append(cleaned)
    # Always keep at least the original so 'multi' never retrieves nothing.
    return subs or [query]


print(decompose("Compare the refund window and the warranty period, and explain the difference."))

## Step 6 — The router: the conductor over Parts 6–10

Now we assemble the pieces into the one function that *is* Adaptive RAG. `route()` classifies the query, then runs **one** of the three pipelines — answering directly, doing a single Part 6 lookup, or decomposing and synthesizing the Part 10 way. The `trace` flag makes the chosen path visible (the demo turns it on).

Read the branches against the essay: route by **complexity**, not by query rewriting (Part 8) and not by knowledge source (Part 10).

In [ ]:
def route(query: str, store, classify, trace=True) -> str:
    """Classify the query, then run the matching pipeline (none/single/multi)."""
    def log(msg):
        if trace:
            print(msg)

    complexity = classify(query)
    log(f'\nQUERY: "{query}"')
    log(f"  classify_complexity -> {complexity}")

    if complexity == "none":
        log("  route -> NO RETRIEVAL (answer directly)")
        return _direct_answer(query)

    if complexity == "single":
        log("  route -> SINGLE-STEP retrieve -> generate (Part 6)")
        retrieved = store.retrieve(query, k=1)
        text, score = retrieved[0]
        log(f"    retrieved (top-1, score={score:.2f}): {text[:42]}...")
        return generate(retrieved)

    # complexity == "multi": decompose -> retrieve per part -> synthesize.
    log("  route -> MULTI-STEP decompose -> retrieve per part -> synthesize (Part 10)")
    subs = decompose(query)
    pieces = []
    for i, sub in enumerate(subs, start=1):
        retrieved = store.retrieve(sub, k=1)
        text, score = retrieved[0]
        log(f'    sub-query {i}: "{sub}"')
        log(f"      retrieved (top-1, score={score:.2f}): {text[:42]}...")
        pieces.append((sub, text))
    lines = "\n".join(f"          - {sub}: {text}" for sub, text in pieces)
    return "Putting the pieces together:\n" + lines


print("route() ready.")

## Step 7 — The demo: six queries, three routes

Now route six queries — two per class — over the support KB, printing the chosen path and the answer for each. Watch the conductor send the greetings down the no-retrieval path (no embedder call at all), the two facts down a single Part 6 lookup, and the two comparisons down the Part 10 decompose-and-synthesize path.

In [ ]:
examples = [
    "hi there",                                                              # none
    "thanks",                                                                # none
    "What is our refund window?",                                            # single
    "How do I fix the E-4042 error?",                                        # single
    "Compare the refund window and the warranty period, and explain the difference.",  # multi
    "What is the difference between the refund window and the warranty period?",        # multi
]

tally = {"none": 0, "single": 0, "multi": 0}
for q in examples:
    tally[classify(q)] += 1
    answer = route(q, store, classify, trace=True)
    print(f"  ANSWER: {answer}".replace("\n", "\n  "))

print("\n" + "-" * 70)
print(f"ROUTE TALLY: none={tally['none']}  single={tally['single']}  multi={tally['multi']}")
print("-" * 70)

## Step 8 — What it buys, and the honest caveats

The payoff is straightforward: the greetings never paid for a retrieval they didn't need, and the comparisons were not crammed into a single lookup that couldn't answer them. One pipeline tuned for the average query is wrong for both ends of the distribution; the classifier lets each query pay only for the machinery it needs.

The numbers Adaptive RAG is sold on — roughly **~35% lower latency** and **~28% lower cost** on a mixed query stream — come from a 2026 vendor production report, **not** from the original paper. Treat them as **indicative direction**, not measured fact: your savings depend entirely on your query mix. And the classifier is itself a **failure surface**: a misroute (sending a hard query down the `single` path, say) means under-retrieval and a worse answer. So grade the router like any other component (Part 11), and keep the routes simple enough to debug.

## What you learned

- A **fixed** pipeline either **over-serves** easy queries (paying for retrieval they don't need) or **under-serves** hard ones (one lookup can't answer a comparison). **Adaptive RAG** reads each query first and routes it.
- `classify_complexity(query)` is the heart of it — a tiny, fully legible classifier returning `none` / `single` / `multi`. In production it's a small trained model; we kept it as transparent rules so every routing decision is inspectable.
- `route()` is the **conductor** over the pipelines we already built: `none` answers directly, `single` runs the **Part 6** retrieve → generate, `multi` runs the **Part 10** decompose → retrieve → synthesize.
- This is route-by-**complexity** — distinct from **Part 8** (transform the query) and **Part 10** (route by knowledge *source*). The classifier is a failure surface; the headline latency/cost wins are **indicative/vendor**, not hard claims.
- Everything ran **fully offline**: the real classifier / embedder / LLM paths are the headline code, with transparent deterministic stand-ins so every cell executes with no network and no API key.

### The close of the Frontier Track

That's the end of the road. The core series ran retrieve → generate to a production-grade finish in **Part 12**; the Frontier Track then picked up three 2026 advances — late-interaction retrieval (Part 13), context-aware chunking (Part 14), and adaptive routing here in Part 15 — each building on what you already knew. There is no Part 16. The capstone checklist still lives in Part 12, and the best next step is the obvious one: **go build**. Point the conductor at your own corpus, watch where it misroutes, and tune from there.